In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 📌 Title
 👉 Titanic Survival Prediction: Machine Learning Pipeline 🤖 

# 📥 1. Import Libraries

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

# 📂 2. Load Dataset

In [5]:
df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

print("📊 Dataset Shape:", df.shape)
df.head()

📊 Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 🔍 3. Basic Info

In [6]:
print("Columns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
df.describe()

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing Values:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# 🧹 4. Data Cleaning

In [7]:
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"], errors="ignore")

print("✅ Missing Values After Cleaning:\n", df.isnull().sum())
df.head()

✅ Missing Values After Cleaning:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


# ⚙️ 5. Feature Engineering

In [8]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = np.where(df["FamilySize"] == 1, 1, 0)
df["Fare_log"] = np.log1p(df["Fare"])

# Title extraction
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

print("📊 Feature Engineering Output:")
df[["FamilySize", "IsAlone", "Fare_log", "Title"]].head()

📊 Feature Engineering Output:


,FamilySize,IsAlone,Fare_log,Title
0,2,0,2.110213,Mr
1,2,0,4.280593,Mrs
2,1,1,2.188856,Miss
3,2,0,3.990834,Mrs
4,1,1,2.202765,Mr


# 🔢 6. Encoding

In [9]:
df["Sex"] = df["Sex"].map({"male": 1, "female": 0})
df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

title_map = {title: i for i, title in enumerate(df["Title"].unique())}
df["Title"] = df["Title"].map(title_map)

print("📊 Encoded Columns:")
df[["Sex", "Embarked", "Title"]].head()

📊 Encoded Columns:


,Sex,Embarked,Title
0,1,0,0
1,0,1,1
2,0,0,2
3,0,0,1
4,1,0,0


# 🎯 7. Feature Selection

In [10]:
features = ["Pclass", "Age", "Fare_log", "FamilySize", "Sex", "Embarked", "Title"]
X = df[features]
y = df["Survived"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)
X.head()

X Shape: (891, 7)
y Shape: (891,)


,Pclass,Age,Fare_log,FamilySize,Sex,Embarked,Title
0,3,22.0,2.110213,2,1,0,0
1,1,38.0,4.280593,2,0,1,1
2,3,26.0,2.188856,1,0,0,2
3,1,35.0,3.990834,2,0,0,1
4,3,35.0,2.202765,1,1,0,0


# 🔀 8. Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (712, 7)
Testing Shape: (179, 7)


# 🤖 9. Model Training


# Logistic Regression

In [12]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.7821229050279329


# Decision Tree

In [13]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))

Decision Tree Accuracy: 0.7486033519553073


# Random Forest

In [14]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.8324022346368715


# 📊 10. Evaluation

In [15]:
print("📊 Confusion Matrix (Logistic Regression):\n")
print(confusion_matrix(y_test, lr_pred))

📊 Confusion Matrix (Logistic Regression):

[[96 22]
 [17 44]]


# 🏁 11. Conclusion
* Random Forest usually performs best
* Feature engineering improved results
* Model comparison helps choose best algorithm